In [8]:
import numpy as np
import copy

class Instancia():
    def __init__(self, x = int, y = int):
        self.x = x
        self.y = y
        self.inicio = {'x': x/2, 'y': 0}
        self.meta = {'x': x/2, 'y': y}

    def __str__(self):
        return f"({self.x}, {self.y}, a meta es {self.meta}, la posicion actual es {self.inicio})"



class Estado():
    def __init__(self, x = int, y = int):
        self.x = x
        self.y = y
        self.acciones = self.obtener_acciones()
    def __str__(self):
        return f"(x: {self.x}, y : {self.y})"

    def __eq__(self, otro_estado):
        return self.x == otro_estado.x and self.y == otro_estado.y

    def __hash__(self):
        return hash((self.x, self.y))

    def obtener_acciones(self):
        acciones = []
        movimientos_posibles = [0,1,-1]
        for mov_x in movimientos_posibles:
            for mov_y in movimientos_posibles:
                acciones.append(Accion(mov_x, mov_y)) # Usar la clase Acción
        return acciones




class Accion():
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __str__(self):
        return f"(x: {self.x}, y: {self.y})"

    def __eq__(self, otra_accion):
        return self.x == otra_accion.x and self.y == otra_accion.y

    def __hash__(self):
        return hash((self.x, self.y))





class Proceso():
    def __init__(self, instancia = Instancia):
        self.instancia = instancia

    def obtener_Estado_inicial(self):
        return Estado(self.instancia.inicio['x'], self.instancia.inicio['y'])

    def transicion(self, accion: Accion, estado = Estado): # Accion como tipo
        nuevo_estado = copy.deepcopy(estado) # creamos una copia del estado

        nuevo_estado.x += accion.x
        nuevo_estado.y += accion.y

        es_Terminal = self.determinar_recompensa(nuevo_estado)
        if es_Terminal == 1:
            return nuevo_estado, 1000
        elif es_Terminal == 0:
            return nuevo_estado, -0.1
        else:
            return nuevo_estado, -100


    def determinar_recompensa(self, estado = Estado):
        if {'x': estado.x, 'y' : estado.y} == self.instancia.meta: # llegó a la meta
            return 1
        elif 0<= estado.x <= self.instancia.x and 0<= estado.y <= self.instancia.y: # Sigue en una posicion valida
            return 0
        else: # Quiere decir que se salio de los limites
            return -1


    def Es_terminal(self, estado):
        return self.determinar_recompensa(estado) == 1 or self.determinar_recompensa(estado) == -1


class Politica():
    def __init__(self, proceso = Proceso, instancia = Instancia):
        self.proceso = proceso
        self.instancia = instancia
        self.q = None # debemos inicializar los q-valores
        self.sa_n = None # debemos inicializar el número de veces que aparece cada estado-acción
        self.inicializar_q()
        self.inicializar_sa_n()

    def inicializar_sa_n(self):
        posiciones_posibles_x = np.arange(0,self.instancia.x+1)
        posiciones_posibles_y = np.arange(0,self.instancia.y+1)
        estados_posibles = [ Estado(x,y) for x in posiciones_posibles_x for y in posiciones_posibles_y ]
        acciones_posibles = [ Accion(x,y) for x in [-1,0,1] for y in [-1,0,1]] # Usar la clase Accion
        self.sa_n = {estado: {accion: 0  for accion in acciones_posibles} for estado in estados_posibles}

    def inicializar_q(self):
        posiciones_posibles_x = np.arange(0,self.instancia.x+1)
        posiciones_posibles_y = np.arange(0,self.instancia.y+1)
        estados_posibles = [ Estado(x,y) for x in posiciones_posibles_x for y in posiciones_posibles_y ]
        acciones_posibles = [ Accion(x,y) for x in [-1,0,1] for y in [-1,0,1]] # Usar la clase Accion
        self.q = {estado: {accion: 0  for accion in acciones_posibles} for estado in estados_posibles}


    def ejecturar_MC_On_Policy(self, num_episodios = int):
        '''"Método que ejecuta el algoritmo de MonteCarlo on policy"'''
        for i in range(num_episodios):
            trayectoria = self.ejecutar_episodio_e_greedy()
            G = 0
            for t in reversed(trayectoria):
                G = G*0.9 + t['recompensa']
                n = self.sa_n[t['estado']][t['accion']]
                
                # actualizamos el valor de la q-tabla
                self.q[t['estado']][t['accion']] = (n * self.q[t['estado']][t['accion']] + G) / (n + 1)
                valor_nuevo = self.q[t['estado']][t['accion']]
                # aumentamos el contador de veces que aparece el estado-accion
                self.sa_n[t['estado']][t['accion']] += 1


    def ejecutar_episodio_e_greedy(self):
        estado = self.proceso.obtener_Estado_inicial()
        trayectoria = []

        while True:
            accion = self.elegir_accion_e_greedy(estado)
            nuevo_estado, recompensa = self.proceso.transicion(accion, estado)
            trayectoria.append({'estado':estado, 'accion':accion, 'recompensa': recompensa})

            if self.proceso.Es_terminal(nuevo_estado) == -1 or self.proceso.Es_terminal(nuevo_estado) == 1:
                break
            estado = nuevo_estado
        return trayectoria


    def elegir_accion_e_greedy(self, estado = Estado):
        acciones = estado.acciones
        valor_aleatorio = np.random.rand()
        if valor_aleatorio < 0.2:
            return np.random.choice(acciones)
        
        # ahora elegiremos la que tenga el mayor q valor 
        mayor_q_valor = -np.inf
        accion_elegida = None
        for accion in acciones:
            if self.q[estado][accion] >= mayor_q_valor:
                mayor_q_valor = self.q[estado][accion]
                accion_elegida = accion
        return accion_elegida
    

    def ejecutar_episodio_optimo(self):
        estado = self.proceso.obtener_Estado_inicial()
        trayectoria = []
        while True:
            accion = self.elegir_accion_optima(estado)
            nuevo_estado, recompensa = self.proceso.transicion(accion, estado)
            trayectoria.append({'estado':estado, 'accion':accion, 'recompensa': recompensa})
            if  self.proceso.Es_terminal(nuevo_estado) == True:#self.proceso.Es_terminal(nuevo_estado) == -1 or self.proceso.Es_terminal(nuevo_estado) == 1:
                break
            estado = nuevo_estado
        return trayectoria


    def elegir_accion_optima(self, estado = Estado):
        acciones = estado.acciones
        mayor_q_valor = -np.inf
        accion_elegida = None
        for accion in acciones:
            if self.q[estado][accion] >= mayor_q_valor:
                mayor_q_valor = self.q[estado][accion]
                accion_elegida = accion
        return accion_elegida

In [9]:
instancia = Instancia(x = 10, y = 10)
proceso = Proceso(instancia)
politica = Politica(proceso, instancia)


politica.ejecturar_MC_On_Policy(num_episodios= 300)

In [10]:
trayectoria = politica.ejecutar_episodio_optimo()

for estado_accion in trayectoria:
    print(estado_accion['estado'])
    print(estado_accion['accion'])

sum(estado_accion['recompensa'] for estado_accion in trayectoria)


(x: 5.0, y : 0)
(x: 0, y: 1)
(x: 5.0, y : 1)
(x: 1, y: 1)
(x: 6.0, y : 2)
(x: -1, y: 1)
(x: 5.0, y : 3)
(x: 1, y: 0)
(x: 6.0, y : 3)
(x: -1, y: 1)
(x: 5.0, y : 4)
(x: 1, y: 1)
(x: 6.0, y : 5)
(x: 0, y: 1)
(x: 6.0, y : 6)
(x: 0, y: 1)
(x: 6.0, y : 7)
(x: -1, y: 1)
(x: 5.0, y : 8)
(x: 1, y: 1)
(x: 6.0, y : 9)
(x: -1, y: 1)


999.0

In [4]:
politica.q[proceso.obtener_Estado_inicial()]

{<__main__.Accion at 0x1f38aee90d0>: -100.0,
 <__main__.Accion at 0x1f38aee9690>: 53.117850181119564,
 <__main__.Accion at 0x1f38aee9650>: 168.21459000031481,
 <__main__.Accion at 0x1f38aee9610>: -100.0,
 <__main__.Accion at 0x1f38aee9710>: -15.093502005565647,
 <__main__.Accion at 0x1f38aee9d10>: 4.402140384346062,
 <__main__.Accion at 0x1f38aee9cd0>: -100.0,
 <__main__.Accion at 0x1f38aee9d50>: 57.55569259165525,
 <__main__.Accion at 0x1f38aee9d90>: 110.78675705963332}